In [ ]:
!pip install -q -U datasets huggingface_hub requests

import json
import os
import time
import random
import traceback
from datetime import datetime, timezone

import requests
from datasets import Dataset, load_dataset
from huggingface_hub import HfApi

In [2]:
# ## 1. CONFIG — edit everything in this cell

# ---- Hugging Face ----
from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

SYSTEM_LABEL = "finetuned"          

INPUT_DATASET_REPO = "businessrules/Qwen_base_tuned_exp10_results"  
OUTPUT_DATASET_REPO = "businessrules/exp10_promptB_results"  
# INPUT_DATASET_REPO = "businessrules/GPT4_baseline_results"  
# OUTPUT_DATASET_REPO = "businessrules/gpt4.1_promptB_results"  


ID_COLUMN = "id"                 
OUTPUT_COLUMN = "finetuned_model_prediction"
# OUTPUT_COLUMN = "gpt4_prediction"
SOURCE_CODE_COLUMN = "source_code"  
MAX_ITEMS = None
INPUT_SPLIT = "train"

# ---- OpenRouter ----
OPENROUTER_API_KEY = UserSecretsClient().get_secret("OPENROUTER_API_KEY")

JUDGE_MODELS = [
    "openai/gpt-4o-mini",
    "anthropic/claude-sonnet-4.6",
    "google/gemini-2.5-flash",
]

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
REQUEST_TIMEOUT = 120
TEMPERATURE = 0.0
MAX_TOKENS = 3000

# ---- Checkpointing ----
SAVE_EVERY = 10          # push to the HF output dataset after this many NEW results
MAX_RETRIES = 5          # per API call, with exponential backoff
BASE_BACKOFF_SECONDS = 5


In [3]:
# ## 2. Prompt B — system prompt and user prompt template
SYSTEM_PROMPT = """You are a strict code-to-business-rule auditor. You are given a source code snippet and
a list of business rules a model claims to have extracted from it. Your job is to check,
rule by rule, whether each rule is (a) actually supported by the code, (b) written as a
single atomic requirement, (c) falsifiable/testable, and (d) expressed in business
language rather than leaking code-level artifacts.

You are NOT comparing against any human-written reference — only against the source code
shown to you. Ignore style/formatting quality; focus only on
grounding, atomicity, testability, and terminology.

Be skeptical by default. A rule is only "grounded" if you can point to the specific
lines/logic in the code that produce it. If you cannot locate that logic, the rule is
hallucinated, even if it sounds plausible or is common business sense. Do not give the
benefit of the doubt to rules that merely sound reasonable.

You must evaluate every rule individually before computing any aggregate score. Do not
assign scores first and justify after. Scores must be a deterministic function of your
per-rule judgments (formulas provided below).

Output valid JSON only. No prose outside the JSON object. No markdown code fences."""

USER_PROMPT_TEMPLATE = """Evaluate the following generated business rules against the source code they claim to
be extracted from. Ignore markdown formatting/presentation entirely — score only
grounding, atomicity, testability, and terminology.

SOURCE CODE:
\"\"\"
{source_code}
\"\"\"

GENERATED OUTPUT:
\"\"\"
{generated_output}
\"\"\"

First, extract every individual rule from the generated output into a numbered list
(rule_id starting at 1), using your own paraphrase is not necessary — just identify each
distinct rule statement as written. Then evaluate EACH rule on all four dimensions below
before moving to aggregation.

---

### Per-rule dimension 1: Groundedness (hallucination check)
- grounded: true/false — can you point to specific logic in the source code that
  produces this rule?
- evidence_location: if grounded=true, briefly describe the code construct that supports
  it (e.g. "the if-condition checking booking.status on line handling cancellation").
  If grounded=false, write "not found in source".
- contradicts_code: true/false — does the rule state something that directly conflicts
  with what the code does (stronger than just "unsupported" — actively wrong)?

### Per-rule dimension 2: Atomicity
- atomic: true/false — does the rule express exactly ONE decision/condition-action pair?
  A rule combining two independent business requirements joined by "and"/"additionally"/
  a semicolon into one bullet is NOT atomic, even if each half is individually valid.
  A rule requiring multiple conditions to jointly trigger ONE action (e.g. "if A and B,
  then C") IS atomic — that is a single requirement, not two.
- atomicity_issue: if atomic=false, briefly state what should have been split into
  separate rules.

### Per-rule dimension 3: Testability
- testable: true/false — could a QA engineer or a unit test directly verify this rule
  against system behavior, i.e. is it phrased as a concrete, falsifiable condition
  rather than a vague statement (e.g. "the system should handle bookings well" is NOT
  testable; "a booking cannot be cancelled less than 24 hours before start time" IS
  testable)?
- testability_issue: if testable=false, briefly state why (e.g. "vague/subjective
  language", "no concrete condition given").

### Per-rule dimension 4: Terminology / jargon leakage
- has_code_artifact: true/false — does the rule contain a raw code-level artifact
  instead of a business term? This includes: class/method names, namespace separators
  (e.g. "::", "->"), variable names in camelCase/snake_case, enum/constant literals,
  file paths, or any other identifier a business stakeholder would not recognize.
- offending_terms: list of the exact offending substrings found (empty list if none).

---

After evaluating all rules, compute the following aggregates:

total_rules_count = number of rules listed

hallucination_rate = (count of rules with grounded=false) / total_rules_count
contradiction_count = count of rules with contradicts_code=true

atomicity_ratio = (count of rules with atomic=true) / total_rules_count
testability_ratio = (count of rules with testable=true) / total_rules_count
jargon_free_ratio = (count of rules with has_code_artifact=false) / total_rules_count

Convert each ratio to a 1-5 score using this same formula for all three:
  1 if ratio < 0.2
  2 if ratio < 0.4
  3 if ratio < 0.6
  4 if ratio < 0.85
  5 if ratio >= 0.85

hallucination_score (1-5) is inverted since lower hallucination is better:
  5 if hallucination_rate == 0
  4 if hallucination_rate < 0.15
  3 if hallucination_rate < 0.3
  2 if hallucination_rate < 0.5
  1 if hallucination_rate >= 0.5

---

Calibration reference:
- A score of 5 on any dimension requires zero violating rules found — not "mostly fine."
- Do not default to the middle of the scale. If you are unsure whether a rule is
  grounded, re-read the source code before deciding — do not mark grounded=true out of
  charity.
- contradicts_code should be rare and reserved for genuine factual conflicts, not minor
  phrasing differences.

Return ONLY a JSON object matching the schema below. Every rule you list must appear
with all four dimensions evaluated — do not skip rules or leave fields null.

JSON_OUTPUT_SCHEMA:
{{
  "rules": [
    {{
      "rule_id": 1,
      "rule_text": "A booking cannot be cancelled less than 24 hours before the start time.",
      "groundedness": {{
        "grounded": true,
        "evidence_location": "if-check comparing (start_time - now) < 24h before allowing cancel",
        "contradicts_code": false
      }},
      "atomicity": {{
        "atomic": true,
        "atomicity_issue": null
      }},
      "testability": {{
        "testable": true,
        "testability_issue": null
      }},
      "terminology": {{
        "has_code_artifact": false,
        "offending_terms": []
      }}
    }}
  ],
  "aggregates": {{
    "total_rules_count": 1,
    "hallucination_rate": 0.0,
    "hallucination_score": 5,
    "contradiction_count": 0,
    "atomicity_ratio": 1.0,
    "atomicity_score": 5,
    "testability_ratio": 1.0,
    "testability_score": 5,
    "jargon_free_ratio": 1.0,
    "jargon_free_score": 5
  }},
  "overall_notes": "Free-text summary of the single most significant grounding or terminology issue, 1-2 sentences max."
}}"""


In [ ]:
# ## 3. Load the input dataset (test split)

print(f"Loading {INPUT_DATASET_REPO} split={INPUT_SPLIT} (system={SYSTEM_LABEL}) ...")
input_ds = load_dataset(INPUT_DATASET_REPO, split=INPUT_SPLIT, token=HF_TOKEN)
print(f"Loaded {len(input_ds)} rows. Columns: {input_ds.column_names}")

if ID_COLUMN not in input_ds.column_names:
    print(f"WARNING: id_column '{ID_COLUMN}' not found — falling back to row index as id.")
if OUTPUT_COLUMN not in input_ds.column_names:
    raise ValueError(
        f"OUTPUT_COLUMN '{OUTPUT_COLUMN}' not found in {INPUT_DATASET_REPO}. "
        f"Available columns: {input_ds.column_names}"
    )
if SOURCE_CODE_COLUMN not in input_ds.column_names:
    raise ValueError(
        f"SOURCE_CODE_COLUMN '{SOURCE_CODE_COLUMN}' not found in {INPUT_DATASET_REPO}. "
        f"Available columns: {input_ds.column_names}"
    )
if MAX_ITEMS is not None:
    input_ds = input_ds.select(range(min(MAX_ITEMS, len(input_ds))))
    print(f"MAX_ITEMS set — using only the first {len(input_ds)} rows for this run.")


# ## 4. OpenRouter call + JSON parsing/validation helpers

def call_openrouter(model: str, system_prompt: str, user_prompt: str) -> str:
    """Calls OpenRouter chat completions, returns raw text content. Retries with
    exponential backoff on transient failures (429, 5xx, timeouts)."""
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": model,
        "temperature": TEMPERATURE,
        "max_tokens": MAX_TOKENS,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    
        "response_format": {"type": "json_object"},
    }

    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.post(
                OPENROUTER_URL, headers=headers, json=payload, timeout=REQUEST_TIMEOUT
            )
            if resp.status_code == 429 or resp.status_code >= 500:
                raise RuntimeError(f"HTTP {resp.status_code}: {resp.text[:300]}")
            resp.raise_for_status()
            data = resp.json()
            return data["choices"][0]["message"]["content"]
        except Exception as e:
            last_err = e
            sleep_s = BASE_BACKOFF_SECONDS * (2 ** (attempt - 1)) + random.uniform(0, 2)
            print(f"  [retry {attempt}/{MAX_RETRIES}] {model} call failed: {e}. "
                  f"Sleeping {sleep_s:.1f}s ...")
            time.sleep(sleep_s)
    raise RuntimeError(f"OpenRouter call failed after {MAX_RETRIES} retries: {last_err}")


def strip_code_fences(text: str) -> str:
    t = text.strip()
    if t.startswith("```"):
        t = t.split("\n", 1)[1] if "\n" in t else t
        if t.endswith("```"):
            t = t.rsplit("```", 1)[0]
    return t.strip()


def _ratio_score(ratio: float) -> int:
    if ratio < 0.2:
        return 1
    if ratio < 0.4:
        return 2
    if ratio < 0.6:
        return 3
    if ratio < 0.85:
        return 4
    return 5


def _hallucination_score(rate: float) -> int:
    if rate == 0:
        return 5
    if rate < 0.15:
        return 4
    if rate < 0.3:
        return 3
    if rate < 0.5:
        return 2
    return 1


def recompute_checks(parsed: dict) -> dict:
    """Recomputes total_rules_count / hallucination_rate / hallucination_score /
    contradiction_count / atomicity_ratio / atomicity_score / testability_ratio /
    testability_score / jargon_free_ratio / jargon_free_score from the per-rule
    booleans, per the prompt's own instructions. Flags mismatches vs what the
    model reported in its own 'aggregates' block."""
    issues = []

    rules = parsed.get("rules", [])
    total = len(rules)

    def get_bool(rule, section, field):
        return rule.get(section, {}).get(field)

    grounded_false_count = sum(
        1 for r in rules if get_bool(r, "groundedness", "grounded") is False
    )
    contradiction_count = sum(
        1 for r in rules if get_bool(r, "groundedness", "contradicts_code") is True
    )
    atomic_count = sum(
        1 for r in rules if get_bool(r, "atomicity", "atomic") is True
    )
    testable_count = sum(
        1 for r in rules if get_bool(r, "testability", "testable") is True
    )
    jargon_free_count = sum(
        1 for r in rules if get_bool(r, "terminology", "has_code_artifact") is False
    )

    hallucination_rate = (grounded_false_count / total) if total > 0 else 0.0
    atomicity_ratio = (atomic_count / total) if total > 0 else 0.0
    testability_ratio = (testable_count / total) if total > 0 else 0.0
    jargon_free_ratio = (jargon_free_count / total) if total > 0 else 0.0

    recomputed = {
        "total_rules_count": total,
        "hallucination_rate": hallucination_rate,
        "hallucination_score": _hallucination_score(hallucination_rate),
        "contradiction_count": contradiction_count,
        "atomicity_ratio": atomicity_ratio,
        "atomicity_score": _ratio_score(atomicity_ratio),
        "testability_ratio": testability_ratio,
        "testability_score": _ratio_score(testability_ratio),
        "jargon_free_ratio": jargon_free_ratio,
        "jargon_free_score": _ratio_score(jargon_free_ratio),
    }

    agg = parsed.get("aggregates", {})
    if not isinstance(agg, dict):
        agg = {}
        issues.append("aggregates field malformed")

    for key, recomputed_val in recomputed.items():
        model_val = agg.get(key)
        if isinstance(recomputed_val, float):
            mismatch = (
                model_val is None
                or not isinstance(model_val, (int, float))
                or abs(float(model_val) - recomputed_val) > 1e-6
            )
        else:
            mismatch = model_val != recomputed_val
        if mismatch:
            issues.append(
                f"{key} mismatch: model said {model_val}, recomputed {recomputed_val}"
            )

    if agg.get("hallucination_score") == 5 and grounded_false_count > 0:
        issues.append(
            "CONTRADICTION: hallucination_score=5 but at least one rule has grounded=false"
        )

    return {
        "recomputed_total_rules_count": recomputed["total_rules_count"],
        "recomputed_hallucination_rate": recomputed["hallucination_rate"],
        "recomputed_hallucination_score": recomputed["hallucination_score"],
        "recomputed_contradiction_count": recomputed["contradiction_count"],
        "recomputed_atomicity_ratio": recomputed["atomicity_ratio"],
        "recomputed_atomicity_score": recomputed["atomicity_score"],
        "recomputed_testability_ratio": recomputed["testability_ratio"],
        "recomputed_testability_score": recomputed["testability_score"],
        "recomputed_jargon_free_ratio": recomputed["jargon_free_ratio"],
        "recomputed_jargon_free_score": recomputed["jargon_free_score"],
        "validation_issues": issues,
        "is_consistent": len(issues) == 0,
    }


def judge_one(model: str, source_code: str, generated_output: str) -> dict:
    """Runs Prompt B once and returns a flat result dict (never raises — errors are
    captured in the row so the pipeline keeps going)."""
    user_prompt = USER_PROMPT_TEMPLATE.format(
        source_code=source_code, generated_output=generated_output
    )
    raw = None
    try:
        raw = call_openrouter(model, SYSTEM_PROMPT, user_prompt)
        cleaned = strip_code_fences(raw)
        parsed = json.loads(cleaned)
        checks = recompute_checks(parsed)
        return {
            "status": "ok",
            "raw_response": raw,
            "parsed_json": json.dumps(parsed, ensure_ascii=False),
            "error": None,
            **checks,
        }
    except Exception as e:
        return {
            "status": "error",
            "raw_response": raw,
            "parsed_json": None,
            "error": f"{type(e).__name__}: {e}",
            "recomputed_total_rules_count": None,
            "recomputed_hallucination_rate": None,
            "recomputed_hallucination_score": None,
            "recomputed_contradiction_count": None,
            "recomputed_atomicity_ratio": None,
            "recomputed_atomicity_score": None,
            "recomputed_testability_ratio": None,
            "recomputed_testability_score": None,
            "recomputed_jargon_free_ratio": None,
            "recomputed_jargon_free_score": None,
            "validation_issues": [str(e)],
            "is_consistent": False,
        }

In [ ]:
# ## 5. Resume support — load existing output dataset (if any) and skip done work

def load_existing_results():
    try:
        existing = load_dataset(OUTPUT_DATASET_REPO, split="train", token=HF_TOKEN)
        results = existing.to_list()
        done_keys = {(r["item_id"], r["system"], r["judge_model"]) for r in results}
        print(f"Resuming: found {len(results)} existing results "
              f"({len(done_keys)} unique combos) in {OUTPUT_DATASET_REPO}.")
        return results, done_keys
    except Exception as e:
        print(f"No existing output dataset found (starting fresh): {e}")
        return [], set()


all_results, done_keys = load_existing_results()

# ## 6. Build the full work queue: item x judge_model (for SYSTEM_LABEL)

work_items = []
for idx, row in enumerate(input_ds):
    item_id = row.get(ID_COLUMN, idx) if ID_COLUMN in input_ds.column_names else idx
    generated_output = row.get(OUTPUT_COLUMN)
    source_code = row.get(SOURCE_CODE_COLUMN)
    if generated_output is None or source_code is None:
        continue
    for model in JUDGE_MODELS:
        key = (item_id, SYSTEM_LABEL, model)
        if key in done_keys:
            continue
        work_items.append({
            "item_id": item_id,
            "system": SYSTEM_LABEL,
            "judge_model": model,
            "generated_output": generated_output,
            "source_code": source_code,
        })

print(f"Total work items remaining: {len(work_items)}")


# ## 7. Checkpoint save helper

def save_checkpoint(results):
    if not results:
        return
    ds = Dataset.from_list(results)
    ds.push_to_hub(OUTPUT_DATASET_REPO, token=HF_TOKEN)
    print(f"  [checkpoint] pushed {len(results)} total results to {OUTPUT_DATASET_REPO}")


# ## 8. Main loop — judges every work item, checkpointing every SAVE_EVERY new results

new_since_last_save = 0

try:
    for i, item in enumerate(work_items, start=1):
        print(f"[{i}/{len(work_items)}] item_id={item['item_id']} "
              f"system={item['system']} model={item['judge_model']}")

        result = judge_one(item["judge_model"], item["source_code"], item["generated_output"])
        row = {
            "item_id": item["item_id"],
            "system": item["system"],
            "judge_model": item["judge_model"],
            "timestamp": datetime.now(timezone.utc).isoformat(),
            **result,
            "validation_issues": json.dumps(result["validation_issues"]),
        }
        all_results.append(row)
        new_since_last_save += 1

        if new_since_last_save >= SAVE_EVERY:
            save_checkpoint(all_results)
            new_since_last_save = 0

except KeyboardInterrupt:
    print("Interrupted — saving progress before exiting.")
except Exception:
    print("Unexpected error — saving progress before re-raising.")
    print(traceback.format_exc())
finally:
    if new_since_last_save > 0:
        save_checkpoint(all_results)

print(f"Done. {len(all_results)} total results saved to {OUTPUT_DATASET_REPO}.")

# ## 9. Quick sanity check on judge reliability

inconsistent = [r for r in all_results if not r.get("is_consistent", True)]
print(f"{len(inconsistent)}/{len(all_results)} results had aggregate-vs-recomputed mismatches "
      f"(recomputed by this script, per the prompt's own validation note).")
if inconsistent:
    print("Example issues from the first few:")
    for r in inconsistent[:5]:
        print(f" - item_id={r['item_id']} system={r['system']} model={r['judge_model']}: "
              f"{r['validation_issues']}")